# Meteologica DA Price Forecast — DA Cutoff Revision Analysis (Western Hub)

Inspects how Meteologica PJM **Western Hub** DA price forecasts for the next 7 days evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 12h / 24h / 48h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates.

## 1. Setup & Data Pull

In [29]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

In [30]:
sql_path = Path.cwd() / "meteologica_da_price_forecast_da_cutoff_rto.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"Hub: {df['hub'].unique()}")
df.head(10)

188 rows
Date range: 2026-03-05 to 2026-03-12
Hub: <ArrowStringArray>
['WESTERN']
Length: 1, dtype: str


,forecast_date,hour_ending,hub,cutoff_price,cutoff_execution_dt,exec_dt_12h,price_12h,exec_dt_24h,price_24h,exec_dt_48h,price_48h,delta_12h,delta_24h,delta_48h
0,2026-03-12,1,WESTERN,23.72,2026-03-05 06:55:14,2026-03-04 18:51:39,23.15,2026-03-04 06:52:46,22.83,2026-03-03 01:37:25,17.46,0.57,0.89,6.26
1,2026-03-12,2,WESTERN,27.69,2026-03-05 06:55:14,2026-03-04 18:51:39,27.14,2026-03-04 06:52:46,26.95,2026-03-03 01:37:25,19.62,0.55,0.74,8.07
2,2026-03-12,3,WESTERN,39.64,2026-03-05 06:55:14,2026-03-04 18:51:39,39.36,2026-03-04 06:52:46,40.88,2026-03-03 01:37:25,33.30,0.28,-1.24,6.34
3,2026-03-12,4,WESTERN,49.27,2026-03-05 06:55:14,2026-03-04 18:51:39,49.46,2026-03-04 06:52:46,50.69,2026-03-03 01:37:25,44.52,-0.19,-1.42,4.75
4,2026-03-12,5,WESTERN,39.89,2026-03-05 06:55:14,2026-03-04 18:51:39,39.33,2026-03-04 06:52:46,40.75,2026-03-03 01:37:25,37.78,0.56,-0.86,2.11
5,2026-03-12,6,WESTERN,37.08,2026-03-05 06:55:14,2026-03-04 18:51:39,35.04,2026-03-04 06:52:46,35.42,2026-03-03 01:37:25,35.03,2.04,1.66,2.05
6,2026-03-12,7,WESTERN,36.79,2026-03-05 06:55:14,2026-03-04 18:51:39,34.39,2026-03-04 06:52:46,35.60,2026-03-03 01:37:25,34.57,2.40,1.19,2.22
7,2026-03-12,8,WESTERN,38.90,2026-03-05 06:55:14,2026-03-04 18:51:39,36.61,2026-03-04 06:52:46,36.85,2026-03-03 01:37:25,36.52,2.29,2.05,2.38
8,2026-03-12,9,WESTERN,38.94,2026-03-05 06:55:14,2026-03-04 18:51:39,36.28,2026-03-04 06:52:46,35.67,2026-03-03 01:37:25,35.76,2.66,3.27,3.18
9,2026-03-12,10,WESTERN,37.05,2026-03-05 06:55:14,2026-03-04 18:51:39,34.33,2026-03-04 06:52:46,33.63,2026-03-03 01:37:25,34.77,2.72,3.42,2.28


In [31]:
df.dtypes

forecast_date                  object
hour_ending                     int64
hub                               str
cutoff_price                  float64
cutoff_execution_dt    datetime64[us]
exec_dt_12h            datetime64[us]
price_12h                     float64
exec_dt_24h            datetime64[us]
price_24h                     float64
exec_dt_48h            datetime64[us]
price_48h                     float64
delta_12h                     float64
delta_24h                     float64
delta_48h                     float64
dtype: object

## 1b. Forecast Coverage Summary

One row per forecast date showing the cutoff vintage used, each lookback vintage,
and peak/min cutoff price — a quick reference for which days have full 48h→12h history.

In [32]:
_fmt_dt = lambda s: pd.to_datetime(s).strftime("%Y-%m-%d %H:%M") if pd.notna(s) else "\u2014"

forecast_summary = (
    df.groupby("forecast_date")
    .agg(
        cutoff_exec=("cutoff_execution_dt", "first"),
        exec_12h=("exec_dt_12h", "first"),
        exec_24h=("exec_dt_24h", "first"),
        exec_48h=("exec_dt_48h", "first"),
        peak_cutoff_price=("cutoff_price", "max"),
        min_cutoff_price=("cutoff_price", "min"),
        hours_covered=("hour_ending", "nunique"),
    )
    .reset_index()
    .sort_values("forecast_date")
)

forecast_summary["days_ahead"] = (
    pd.to_datetime(forecast_summary["forecast_date"]) - pd.Timestamp.now().normalize()
).dt.days

for col in ["cutoff_exec", "exec_12h", "exec_24h", "exec_48h"]:
    forecast_summary[col] = forecast_summary[col].apply(_fmt_dt)

forecast_summary["has_12h"] = forecast_summary["exec_12h"] != "\u2014"
forecast_summary["has_24h"] = forecast_summary["exec_24h"] != "\u2014"
forecast_summary["has_48h"] = forecast_summary["exec_48h"] != "\u2014"

display_cols = [
    "forecast_date", "days_ahead", "hours_covered",
    "cutoff_exec", "exec_48h", "exec_24h", "exec_12h",
    "has_48h", "has_24h", "has_12h",
    "peak_cutoff_price", "min_cutoff_price",
]
forecast_summary[display_cols].style.set_caption(
    "Meteologica Western Hub \u2014 DA Price Forecast Date Coverage \u2014 7-Day Forward Window"
).format({
    "peak_cutoff_price": "${:,.2f}",
    "min_cutoff_price": "${:,.2f}",
}).set_properties(**{"text-align": "center"})

,forecast_date,days_ahead,hours_covered,cutoff_exec,exec_48h,exec_24h,exec_12h,has_48h,has_24h,has_12h,peak_cutoff_price,min_cutoff_price
0,2026-03-05,0,22,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$63.10,$25.88
1,2026-03-06,1,24,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$53.72,$22.63
2,2026-03-07,2,23,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$37.57,$18.21
3,2026-03-08,3,23,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$39.07,$18.60
4,2026-03-09,4,24,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$47.15,$20.09
5,2026-03-10,5,24,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$45.29,$19.80
6,2026-03-11,6,24,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$48.93,$20.37
7,2026-03-12,7,24,2026-03-05 06:55,2026-03-03 01:37,2026-03-04 06:52,2026-03-04 18:51,True,True,True,$51.39,$23.72


## 2. Data Validation — Cutoff Vintage Inspection

In [33]:
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-03-05,2026-03-05 06:55:14,06:55
1,2026-03-06,2026-03-05 06:55:14,06:55
2,2026-03-07,2026-03-05 06:55:14,06:55
3,2026-03-08,2026-03-05 06:55:14,06:55
4,2026-03-09,2026-03-05 06:55:14,06:55
5,2026-03-10,2026-03-05 06:55:14,06:55
6,2026-03-11,2026-03-05 06:55:14,06:55
7,2026-03-12,2026-03-05 06:55:14,06:55


In [34]:
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Meteologica Western Hub \u2014 Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [35]:
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("price_12h", lambda x: x.notna().any()),
        has_24h=("price_24h", lambda x: x.notna().any()),
        has_48h=("price_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-03-05,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
1,2026-03-06,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
2,2026-03-07,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
3,2026-03-08,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
4,2026-03-09,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
5,2026-03-10,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
6,2026-03-11,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25
7,2026-03-12,True,True,True,2026-03-04 18:51:39,2026-03-04 06:52:46,2026-03-03 01:37:25


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [36]:
# Overlay 48h -> 24h -> 12h -> cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].sort_values("hour_ending")

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

fig = go.Figure()
for label, col, width in [
    ("48h", "price_48h", 1),
    ("24h", "price_24h", 1.5),
    ("12h", "price_12h", 2),
    ("Cutoff", "cutoff_price", 3),
]:
    fig.add_trace(go.Scatter(
        x=df_latest["hour_ending"],
        y=df_latest[col],
        name=label,
        line=dict(color=colors[label], dash=dashes[label], width=width),
    ))

fig.update_layout(
    height=500,
    template="plotly_dark",
    title_text=f"Meteologica Western Hub \u2014 DA Price Forecast Evolution \u2014 {latest_date}",
    xaxis_title="Hour Ending",
    yaxis_title="DA Price ($/MWh)",
)
fig.show()

In [37]:
# All dates — cutoff (solid) vs 24h lookback (dashed)
dates = sorted(df["forecast_date"].unique())

fig = go.Figure()
for d in dates:
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")
    opacity = 0.3 if d != latest_date else 1.0
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["cutoff_price"],
        name=f"{d} cutoff",
        line=dict(width=2 if d == latest_date else 1),
        opacity=opacity,
    ))
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["price_24h"],
        name=f"{d} 24h",
        line=dict(dash="dash", width=1),
        opacity=opacity * 0.6,
        showlegend=False,
    ))

fig.update_layout(
    height=500, template="plotly_dark",
    title="Meteologica Western Hub \u2014 Cutoff (solid) vs 24h Prior (dashed)",
    xaxis_title="Hour Ending", yaxis_title="DA Price ($/MWh)",
)
fig.show()

## 4. Forecast Deltas — Price Changes at Each Lookback

In [38]:
# Grouped bar: delta_12h, delta_24h, delta_48h by hour_ending for latest date
fig = go.Figure()
for col, label, color in [
    ("delta_48h", "48h\u2192Cutoff", "#636EFA"),
    ("delta_24h", "24h\u2192Cutoff", "#EF553B"),
    ("delta_12h", "12h\u2192Cutoff", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=df_latest["hour_ending"], y=df_latest[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title=f"Meteologica Western Hub \u2014 DA Price Forecast Revision Deltas ({latest_date})",
    xaxis_title="Hour Ending", yaxis_title="Delta ($/MWh)",
    template="plotly_dark", height=450,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [39]:
# Heatmap: 24h delta across all dates x hours
pivot_24h = df.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_24h",
)

fig = px.imshow(
    pivot_24h.values,
    x=[f"HE{int(c)}" for c in pivot_24h.columns],
    y=[str(d) for d in pivot_24h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Meteologica Western Hub \u2014 24h DA Price Revision ($/MWh) by Date \u00d7 Hour",
    labels={"color": "Delta $/MWh"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. Forecast Evolution by Date — All Vintages

In [40]:
dates = sorted(df["forecast_date"].unique())

fig = make_subplots(
    rows=len(dates), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Western Hub \u2014 {d}" for d in dates],
    vertical_spacing=0.04,
)

for i, d in enumerate(dates, 1):
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")

    for label, col, exec_col, width in [
        ("48h", "price_48h", "exec_dt_48h", 1),
        ("24h", "price_24h", "exec_dt_24h", 1.5),
        ("12h", "price_12h", "exec_dt_12h", 2),
        ("Cutoff", "cutoff_price", "cutoff_execution_dt", 3),
    ]:
        exec_dts = ddf[exec_col].astype(str).values
        forecast_dates = ddf["forecast_date"].astype(str).values
        deltas = np.zeros(len(ddf)) if label == "Cutoff" else ddf[f"delta_{label}"].values
        rank_labels = np.array([label] * len(ddf))

        customdata = np.column_stack([exec_dts, forecast_dates, deltas, rank_labels])

        fig.add_trace(
            go.Scatter(
                x=ddf["hour_ending"],
                y=ddf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[3]}</b> \u2014 Western Hub<br>"
                    "Forecast Date: %{customdata[1]}<br>"
                    "HE: %{x}<br>"
                    "Price: $%{y:,.2f}/MWh<br>"
                    "Execution DT: %{customdata[0]}<br>"
                    "Delta to Cutoff: $%{customdata[2]}/MWh"
                    "<extra></extra>"
                ),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=250 * len(dates),
    template="plotly_dark",
    title_text="Meteologica Western Hub \u2014 DA Price Forecast Evolution by Date",
)
fig.update_yaxes(title_text="Price ($/MWh)")
fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
fig.show()

## 6. Revision Summary Statistics

In [41]:
summary_rows = []
for lookback, col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    vals = df[col].dropna()
    summary_rows.append({
        "Lookback": lookback,
        "Mean ($/MWh)": vals.mean(),
        "Median ($/MWh)": vals.median(),
        "Std ($/MWh)": vals.std(),
        "Min ($/MWh)": vals.min(),
        "Max ($/MWh)": vals.max(),
        "N": len(vals),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Lookback,Mean ($/MWh),Median ($/MWh),Std ($/MWh),Min ($/MWh),Max ($/MWh),N
0,12h,0.435266,0.280,0.833136,-1.38,3.06,188
1,24h,3.334309,2.530,3.639240,-1.42,21.94,188
2,48h,5.810000,5.435,3.782074,-0.93,26.12,188


In [42]:
# Per-date summary: avg revision direction across all hours
date_summary = (
    df.groupby("forecast_date")
    .agg(
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        mean_delta_48h=("delta_48h", "mean"),
        peak_cutoff_price=("cutoff_price", "max"),
    )
    .reset_index()
)

fig = px.bar(
    date_summary,
    x="forecast_date",
    y="mean_delta_24h",
    title="Meteologica Western Hub \u2014 Mean 24h DA Price Revision ($/MWh) by Date",
    labels={"mean_delta_24h": "Avg 24h Delta ($/MWh)", "forecast_date": "Forecast Date"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.update_layout(height=450)
fig.show()

In [43]:
# Heatmaps for all lookbacks (12h, 24h, 48h)
for lookback, delta_col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    pivot = df.pivot_table(
        index="forecast_date", columns="hour_ending", values=delta_col,
    )
    if pivot.empty:
        continue

    fig = px.imshow(
        pivot.values,
        x=[f"HE{int(c)}" for c in pivot.columns],
        y=[str(d) for d in pivot.index],
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"Meteologica Western Hub \u2014 {lookback} DA Price Revision ($/MWh) by Date \u00d7 Hour",
        labels={"color": "Delta $/MWh"},
        template="plotly_dark",
    )
    fig.update_layout(height=400)
    fig.show()